# Constructing a RAG System and Enhancing it with HyDE

This notebook builds a baseline RAG workflow and improves retrieval quality with HyDE for question answering over a technical document.

## Recommended Hardware

This notebook can run on the following hardware or remote resources

✅ AMD Instinct™ Accelerators  
✅ AMD Radeon™ RX/PRO Graphics Cards  
✅ AMD EPYC™ Processors  
✅ AMD Ryzen™ (AI) Processors  

[![Open in AMD Developer Cloud](https://img.shields.io/badge/Open_in_AMD_Developer_Cloud-000000?logo=amd&logoSize=auto)](https://amd-ai-academy.com/github/AMDResearch/aup-ai-tutorials/blob/main/rag/03.rag-pipeline-HyDE.ipynb)  


## Software Environment

Install ROCm on your system

| Linux | Windows |
|-------|---------|
| [Install PyTorch](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/install/quick-start.html) | [PyTorch on Windows](https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/windows/install-pytorch.html)|
| [Install Docker container](https://amdresearch.github.io/aup-ai-tutorials//env/env-gpu.html) | |

## Goals

- Build a basic RAG pipeline using a real technical document
- Compare standard retrieval with HyDE-based retrieval
- Understand how prompt design affects retrieval quality
- Read and explain LangChain LangChain Expression Language pipelines built with `|` and `RunnablePassthrough`

### Install Dependencies

Install the package dependencies needed for this notebook or series of notebooks.

First, get the `aup_config.py` script locally if needed. Then install the dependencies (`aup_setup()`). This step may take a few minutes and only needs to be done once.

In [1]:
![ -f aup_config.py ] || wget https://raw.githubusercontent.com/AMDResearch/aup-ai-tutorials/refs/heads/main/ai-agents/aup_config.py

In [2]:
from aup_config import aup_setup
aup_setup()

## Build the RAG Pipeline

This section explains how to configure and build the RAG pipeline

### Setup Indexing and the Query Engine
Import the necessary libraries

In [3]:
!pip install -U langchain-text-splitters langchain-ollama

In [4]:
import os
import re

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

from langchain_community.vectorstores import FAISS

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings

/tmp/ipykernel_167/1414924908.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


### Load Medicine Knowledge

Since you already created medicine_info.txt, use it.

In [5]:
loader = TextLoader("medicine_data/medicine_info.txt")

docs = loader.load()

print(f"Documents Loaded: {len(docs)}")

Documents Loaded: 1


### Chunk the Document

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=32
)

documents_split = text_splitter.split_documents(docs)

print(len(documents_split))

1


### Embeddings

In [7]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=32)
documents_split = text_splitter.split_documents(docs)
print(f"Number of documents after splitting: {len(documents_split)}")

Number of documents after splitting: 1


### Embeddings

Instantiate the embedding model that feeds the documents into the vector database

In [9]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

### Create FAISS Vector Database

Instantiate a Chroma vector database with the document chunks

In [10]:
vectordb = FAISS.from_documents(
    documents_split,
    embeddings
)

retriever = vectordb.as_retriever()

### Load Llama 3.1

In [11]:
model = ChatOllama(
    model="llama3.1:8b",
    temperature=0
)

## RAG System

In this section, we connect retrieval, prompt formatting, and generation into one LangChain pipeline using **LangChain Expression Language (LCEL)**.

**LCEL basics.** The pipe operator `|` chains steps so the output of one becomes the input of the next, similar to a Unix pipe. For example, `retriever | format_docs` means "call the retriever, then pass its result to `format_docs`."

**`RunnablePassthrough`.** A `RunnablePassthrough()` is a no-op runnable: it receives an input and forwards it unchanged. It is useful inside dictionary steps where some keys need processing (e.g., retrieval) and another key just needs to carry the original input through. Without it, the original input would be consumed by the previous step and unavailable downstream.

In [12]:
template = """
You are an AI Medicine Assistant.

Answer ONLY using the medicine information below.

If the answer is not present in the context,
say

"I don't have enough information."

Context:

{context}

Question:

{question}

Answer:
"""

### Prompt Template

In [13]:
custom_rag_prompt = PromptTemplate.from_template(template)

### Format Retrieved Documents

In [14]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

### Baseline LCEL Pipeline

The chain below is read left to right through the `|` operators:

1. **Dictionary step** — builds the two variables that `custom_rag_prompt` expects:
   - `"context"`: runs `retriever | format_docs` — the retriever fetches relevant chunks, then `format_docs` joins them into a single string.
   - `"question"`: uses `RunnablePassthrough()` to forward the original query string unchanged. Without this, the raw query would not be available for the prompt's `{question}` placeholder.
2. **`custom_rag_prompt`** — inserts the `context` and `question` values into the prompt template.
3. **`model`** — sends the filled prompt to the LLM.
4. **`StrOutputParser()`** — extracts the plain-text content from the model's response object.

In [15]:
rag_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()} |
             custom_rag_prompt | model | StrOutputParser())

Then test it with medicine-related questions, for example:

In [30]:
query = "What are the uses of Paracetamol?"

response = rag_chain.invoke(query)

print(response)

Relieves fever and mild to moderate pain.


In [28]:
query = "What are the side effects of Ibuprofen?"

In [33]:
results = retriever.invoke(query)

for doc in results:
    print(doc.page_content)
    print("-" * 60)

Medicine: Paracetamol

Uses:
Relieves fever and mild to moderate pain.

Dosage:
500 mg every 4–6 hours.

Side Effects:
Nausea
Vomiting
Rash
Liver damage in overdose

Precautions:
Do not exceed 4 grams per day.
Avoid alcohol while taking this medicine.

Medicine: Aspirin

Uses:
Pain relief
Reduces fever
Blood thinner

Dosage:
75–325 mg as prescribed.

Side Effects:
Stomach irritation
Bleeding
Heartburn

Precautions:
Avoid if allergic to aspirin.
------------------------------------------------------------


## RAG System + HyDE

### Hypothetical Document Generation

Generate a hypothetical answer document to improve retrieval for difficult queries.
Then use that generated document as the retrieval query so the retriever can find more relevant chunks.

In [34]:
template = """
You are a medical expert.

Write a detailed medicine document about:

{user_query}

The document should contain:

Uses

Dosage

Side Effects

Warnings

Precautions

Interactions

Do not answer the user directly.
Generate only the hypothetical document.
"""

### HyDE Prompt Template

In [35]:
prompt_hyde = ChatPromptTemplate.from_template(template)

### HyDE Generation Pipeline

`generate_docs_for_retrieval` is a short pipeline that turns the user query into a hypothetical document.
This synthetic document is not the final answer; it is an intermediate text used to improve retrieval.

In [36]:
generate_docs_for_retrieval = (
    prompt_hyde
    | model
    | StrOutputParser()
)

### Generate Hypothetical Document

In [37]:
hyde_docs = generate_docs_for_retrieval.invoke(
    {
        "user_query": query
    }
)

print(hyde_docs)

**MEDICAL DOCUMENT**

**PARACETAMOL: A REVIEW OF ITS USES, DOSAGE, SIDE EFFECTS, WARNINGS, PRECAUTIONS, AND INTERACTIONS**

**DOCUMENT ID:** PMD-001
**DATE:** March 2023
**AUTHOR:** [Medical Expert]
**REVIEWER:** [Peer Reviewer]

**USES:**

Paracetamol is a widely used analgesic and antipyretic medication. Its uses include:

1. **Relief of mild to moderate pain**: Paracetamol is effective in relieving pain associated with headaches, toothaches, menstrual cramps, and other minor aches.
2. **Fever reduction**: Paracetamol helps reduce fever in patients with viral infections, such as the common cold or flu.
3. **Menstrual relief**: Paracetamol can help alleviate symptoms of dysmenorrhea (painful menstruation).
4. **Post-operative pain management**: Paracetamol is often used to manage pain after surgical procedures.

**DOSE AND ADMINISTRATION:**

The recommended dose and administration of paracetamol vary depending on the patient's age, weight, and medical condition:

* **Adults**: 500-100

### HyDE Retrieval

`retrieval_chain = generate_docs_for_retrieval | retriever` composes two steps into one flow.
First it generates the hypothetical document, then it uses that text to retrieve supporting chunks from the vector database.

In [38]:
retrieval_chain = generate_docs_for_retrieval | retriever

Invoke the retriever

In [39]:
retrieved_docs = retrieval_chain.invoke(
    {
        "user_query": query
    }
)

### RAG + HyDE

Let's finally build the RAG + HyDE pipeline where the custom prompt is passed to the model and then we filter only text from the model's response

In [40]:
rag_hyde_chain = (
    custom_rag_prompt |
    model |
    StrOutputParser()
)

### Ask

Let's finally invoke the full pipeline

In [41]:
rag_hyde_chain.invoke(
    {
        "context": retrieved_docs,
        "question": query
    }
)

'Relieves fever and mild to moderate pain.'

### End-to-End HyDE Pipeline

`rag_hyde_chain_full` combines every stage into a single LCEL chain. The key new primitive here is **`RunnablePassthrough.assign()`**: it passes the current dictionary through unchanged *and* adds (or overwrites) a key with the result of a sub-chain. Think of it as "keep everything we have so far, and also compute this new field."

Step-by-step walkthrough of the chain:

1. `{"user_query": RunnablePassthrough()}` — wraps the raw input string into a dict `{"user_query": "<the query>"}` so downstream steps can reference it by name.
2. `RunnablePassthrough.assign(hypothetical_doc=...)` — keeps `user_query` and adds a new key `hypothetical_doc` whose value is produced by running `prompt_hyde | model | StrOutputParser()`.
3. `RunnablePassthrough.assign(context=...)` — keeps everything and adds `context` by retrieving chunks with the hypothetical document and formatting them.
4. `RunnablePassthrough.assign(question=...)` — maps `question` from the original `user_query` so the prompt template can use it.
5. `custom_rag_prompt | model | StrOutputParser()` — fills the prompt, calls the model, and extracts the text.

This pattern is useful when you want one callable chain instead of manually invoking each stage.

In [42]:
rag_hyde_chain_full = (
    {"user_query": RunnablePassthrough()} |
    RunnablePassthrough.assign(
        hypothetical_doc=prompt_hyde | model | StrOutputParser()
    ) |
    RunnablePassthrough.assign(
        context=lambda x: format_docs(
            retriever.invoke(x["hypothetical_doc"])
        )
    ) |
    RunnablePassthrough.assign(
        question=lambda x: x["user_query"]
    ) |
    custom_rag_prompt |
    model |
    StrOutputParser()
)

Let's invoke the HyDE pipeline

In [43]:
response = rag_hyde_chain_full.invoke(
    "What are the side effects of Paracetamol?"
)

print(response)

The side effects of Paracetamol are: Nausea, Vomiting, Rash, and Liver damage in overdose.


## Exercises for the Reader

- Change the `chunk_size` and re-run both pipelines. What do you observe?
- Increase/Decrease the number of retrieved documents. What do you observe?
- Rewrite the HyDE prompt in a way that the hypothetical document is written as bullet-points. What do you observe?

## References

<div class="alert alert-block alert-info">
<ul>
    <li><a href="https://arxiv.org/abs/2005.11401">Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks</a></li>
    <li><a href="https://arxiv.org/abs/2212.10496">Precise Zero-Shot Dense Retrieval without Relevance Labels</a></li>
    <li><a href="https://blog.langchain.com/langchain-expression-language/">LangChain Expression Language</a></li>
    <li><a href="https://reference.langchain.com/python/langchain-classic/schema/runnable">LCEL runnable</a></li>
    <li><a href="https://rocm.docs.amd.com/projects/ai-developer-hub/en/v3.1/notebooks/inference/rag_ollama_llamaindex.html">RAG System with LlamaIndex</a></li>
</ul>
</div>

## Conclusions

In this notebook, we explored a traditional RAG system and an enhanced RAG system with Hypothetical Document Embeddings

---

[AMD University Program](https://www.amd.com/aup)

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.

SPDX-License-Identifier: MIT